In [1]:
import pandas as pd

dataset = pd.read_csv("../data/heart_disease.csv")
dataset.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0,fixed,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3,normal,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2,reversible,0
3,37,1,3,130,250,0,0,187,0,3.5,3,0,normal,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0,normal,0


In [2]:
dataset.isna().sum()[lambda s: s > 0], dataset.thal.value_counts()

(Series([], dtype: int64),
 thal
 normal        168
 reversible    115
 fixed          18
 1               1
 2               1
 Name: count, dtype: int64)

In [3]:
y = dataset.pop("target")
X = dataset

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y,
)
y.value_counts()

target
0    220
1     83
Name: count, dtype: int64

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = ["cp", "restecg", "slope", "thal"]
numeric_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    [
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("selectkbest", SelectKBest(score_func=f_classif)),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

estimator = GridSearchCV(
    pipeline,
    param_grid={
        "selectkbest__k": list(range(1, 21)),
        "classifier__C": [0.01, 0.1, 1, 10],
    },
    cv=5,
    scoring="balanced_accuracy",
)
estimator.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('standardscaler',
                                                                                          StandardScaler())]),
                                                                         ['age',
                                                                          'sex',
                                                                          'trestbps',
                                                                          'chol',
                                                                          'fbs',
                                                                          'thalach',
                                                                          'exang',
                                                                          'oldpeak',
                                                                          'ca']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['cp',
                                                                          'restecg',
                                                                          'slope',
                                                                          'thal'])])),
                                       ('selectkbest', SelectKBest()),
                                       ('classifier',
                                        LogisticRegression(max_iter=1000))]),
             param_grid={'classifier__C': [0.01, 0.1, 1, 10],
                         'selectkbest__k': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11,
                                            12, 13, 14, 15, 16, 17, 18, 19,
                                            20]},
             scoring='balanced_accuracy')

In [5]:
best = estimator.best_estimator_
feature_names = best.named_steps["preprocessor"].get_feature_names_out()
selected = best.named_steps["selectkbest"].get_support()
estimator.best_params_, estimator.best_score_, list(feature_names[selected])

({'classifier__C': 10, 'selectkbest__k': 7},
 0.8002331002331001,
 ['num__thalach',
  'num__oldpeak',
  'num__ca',
  'cat__cp_4',
  'cat__slope_1',
  'cat__thal_normal',
  'cat__thal_reversible'])

In [6]:
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

y_pred_test = estimator.predict(X_test)
balanced_accuracy_score(y_test, y_pred_test), confusion_matrix(y_test, y_pred_test)

(0.7913419913419913,
 array([[53,  2],
        [ 8, 13]], dtype=int64))

In [7]:
import os
import pickle

os.makedirs("../submission", exist_ok=True)

with open("../submission/estimator.pkl", "wb") as file:
    pickle.dump(estimator, file)